# Ticket 319 abs AUC 재산출 + correlation pruning

065547 의 raw lv3 AUC 를 abs AUC 로 변환 → cutoff `abs_auc >= 0.65` → correlation pruning (`|r| >= 0.85`, abs AUC 높은 거 keep) → XGBoost 재학습.

- abs_auc = |raw_auc - 0.5| + 0.5
- 065547 baseline (2 feature, overall AUC 1.0, lv3 AUC 0.96) 과 비교
- mouse_stop_segment_count (raw lv3_auc = 0.0) NaN imputation 아티팩트 진단 포함
- 산출물: services/ai/train/model_experiment/runs/ticket_319_abs_<timestamp>/

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

try:
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    elif hasattr(sys.stdout, "buffer"):
        sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8")
except Exception:
    pass


RANDOM_STATE = 42
TEST_SIZE = 0.2
ABS_AUC_CUTOFF = 0.65
CORR_CUTOFF = 0.85

FEATURES_320 = [
    "time_to_first_click_ms",
    "inter_click_interval_ms",
    "pre_click_mousemove_count",
    "mouse_total_travel_distance_px",
    "mouse_avg_speed_px_per_ms",
    "mouse_max_speed_px_per_ms",
    "mouse_speed_change_mean",
    "mouse_acceleration_mean",
    "mouse_jerk_mean",
    "mouse_path_straightness_score",
    "mouse_path_curvature_mean",
    "mouse_direction_change_count",
    "mouse_overshoot_flag",
    "mouse_hover_dwell_time_ms",
    "mouse_stop_segment_count",
    "mousemove_event_rate",
]

# 065547 baseline 2-feature XGBoost 결과 (비교 anchor)
BASELINE_065547 = {
    "feature_count": 2,
    "overall_auc": 1.0,
    "lv3_auc": 0.96,
    "features": ["time_to_first_click_ms", "mouse_path_straightness_score"],
}


def source_group(trial_id, label, algorithm_type):
    if 900001 <= trial_id <= 909999:
        return "lv2_human" if label == "human" else "lv2_macro"
    if 910001 <= trial_id <= 910500:
        return "balabit"
    if 930001 <= trial_id <= 930050 or algorithm_type == "lv3_balabit_kde":
        return "lv3_balabit_kde"
    if 940001 <= trial_id <= 949999 or algorithm_type == "lv4_aggressive":
        return "lv4_aggressive"
    return "unknown"


def load_existing_trials(behavior_dir):
    rows = []
    for path in sorted(behavior_dir.glob("trial_*.json")):
        trial = json.loads(path.read_text(encoding="utf-8"))
        trial_id = int(trial.get("trialId") or path.stem.split("_", 1)[1])
        label = trial.get("label")
        metrics = trial.get("metrics") or {}
        algorithm_type = trial.get("algorithm_type")
        row = {
            "trial_id": trial_id,
            "label": label,
            "label_int": 1 if label == "macro" else 0,
            "source_group": source_group(trial_id, label, algorithm_type),
            "algorithm_type": algorithm_type,
        }
        for feature in FEATURES_320:
            row[feature] = metrics.get(feature)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("trial_id").reset_index(drop=True)


def load_today_macro(path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        features = obj.get("features") or {}
        row = {
            "trial_id": int(obj.get("trialID")),
            "label": "macro",
            "label_int": 1,
            "source_group": "today_macro",
            "algorithm_type": "today_macro_after53",
        }
        for feature in FEATURES_320:
            row[feature] = features.get(feature)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("trial_id").reset_index(drop=True)


def build_condition_frames(existing, today_macro):
    baseline = existing[existing["source_group"].isin({"lv2_human", "lv2_macro", "balabit"})].copy()
    plus_today = pd.concat([baseline, today_macro], ignore_index=True)
    lv3_eval = existing[existing["source_group"].isin({"balabit", "lv3_balabit_kde"})].copy()
    return {"baseline": baseline, "+today": plus_today, "lv3_eval": lv3_eval}


def split_frame(df):
    train_idx, test_idx = train_test_split(
        df.index, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["label_int"]
    )
    return df.loc[train_idx].copy(), df.loc[test_idx].copy()


def make_xgb():
    return XGBClassifier(
        n_estimators=400,
        max_depth=3,
        learning_rate=0.03,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=1,
        tree_method="hist",
    )


def to_abs_auc(raw_auc):
    return abs(raw_auc - 0.5) + 0.5


def diagnose_stop_segment(existing, today_macro):
    combined = pd.concat([existing, today_macro], ignore_index=True)
    groups = {
        "human": combined[combined["label"] == "human"],
        "lv2_macro": combined[combined["source_group"] == "lv2_macro"],
        "lv3_balabit_kde": combined[combined["source_group"] == "lv3_balabit_kde"],
        "today_macro": combined[combined["source_group"] == "today_macro"],
    }
    out = {}
    for name, g in groups.items():
        col = g["mouse_stop_segment_count"]
        non_null = col.dropna()
        out[name] = {
            "n": int(len(g)),
            "non_null_count": int(non_null.size),
            "null_count": int(col.isna().sum()),
            "non_null_ratio": float(non_null.size / len(g)) if len(g) else 0.0,
            "median": float(non_null.median()) if non_null.size else None,
            "min": float(non_null.min()) if non_null.size else None,
            "max": float(non_null.max()) if non_null.size else None,
        }
    return out


def prune_by_correlation(pool, train_df, abs_auc_map, corr_cutoff):
    if len(pool) < 2:
        return list(pool), []
    df_feat = train_df[pool].copy()
    corr = df_feat.corr().abs()
    pairs = []
    for i, f1 in enumerate(pool):
        for f2 in pool[i + 1 :]:
            r = corr.loc[f1, f2]
            if pd.notna(r) and r >= corr_cutoff:
                pairs.append((f1, f2, float(r)))
    pairs.sort(key=lambda x: x[2], reverse=True)
    kept = list(pool)
    log = []
    for f1, f2, r in pairs:
        if f1 not in kept or f2 not in kept:
            continue
        a1 = abs_auc_map.get(f1, 0.5)
        a2 = abs_auc_map.get(f2, 0.5)
        if a1 >= a2:
            kept.remove(f2)
            log.append({"dropped": f2, "kept": f1, "abs_corr": r,
                        "kept_abs_auc": a1, "dropped_abs_auc": a2})
        else:
            kept.remove(f1)
            log.append({"dropped": f1, "kept": f2, "abs_corr": r,
                        "kept_abs_auc": a2, "dropped_abs_auc": a1})
    return kept, log


def train_xgb(condition, pool, train_df, test_df, lv3_eval):
    if not pool:
        return {"condition": condition, "feature_count": 0,
                "overall_auc": float("nan"), "lv3_auc": float("nan"), "features": []}
    model = make_xgb()
    model.fit(train_df[pool], train_df["label_int"].values)
    overall_pred = model.predict_proba(test_df[pool])[:, 1]
    lv3_pred = model.predict_proba(lv3_eval[pool])[:, 1]
    return {
        "condition": condition,
        "feature_count": len(pool),
        "overall_auc": float(roc_auc_score(test_df["label_int"].values, overall_pred)),
        "lv3_auc": float(roc_auc_score(lv3_eval["label_int"].values, lv3_pred)),
        "features": pool,
    }


def fmt_or_none(value):
    if value is None:
        return "—"
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


def main():
    try:
        ai_root = Path(__file__).resolve().parents[2]
    except NameError:
        ai_root = Path.cwd().resolve().parents[1]

    parser = argparse.ArgumentParser()
    parser.add_argument("--behavior-dir", type=Path,
                        default=ai_root / "data" / "behavior")
    parser.add_argument("--today-macro-jsonl", type=Path,
                        default=ai_root / "data" / "behavior_exports" / "macro_today_after53.jsonl")
    parser.add_argument("--source-univariate", type=Path,
                        default=ai_root / "train" / "model_experiment" / "runs" /
                                "ticket_319_20260511_065547" / "univariate_compare.json")
    parser.add_argument("--run-dir", type=Path,
                        default=ai_root / "train" / "model_experiment" / "runs" /
                                f"ticket_319_abs_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    args = parser.parse_args(args=[])

    args.run_dir.mkdir(parents=True, exist_ok=False)

    existing = load_existing_trials(args.behavior_dir)
    today_macro = load_today_macro(args.today_macro_jsonl)
    frames = build_condition_frames(existing, today_macro)

    baseline_train, baseline_test = split_frame(frames["baseline"])
    today_train, today_test = split_frame(frames["+today"])
    lv3_eval = frames["lv3_eval"]

    # -- 1) abs AUC from existing raw univariate
    raw_uni = json.loads(args.source_univariate.read_text(encoding="utf-8"))
    abs_rows = []
    for row in raw_uni:
        feature = row["feature"]
        b_raw = float(row["baseline_lv3_auc"])
        t_raw = float(row["+today_lv3_auc"])
        abs_rows.append({
            "feature": feature,
            "baseline_raw_lv3_auc": b_raw,
            "baseline_abs_lv3_auc": to_abs_auc(b_raw),
            "today_raw_lv3_auc": t_raw,
            "today_abs_lv3_auc": to_abs_auc(t_raw),
            "baseline_overall_auc": float(row["baseline_overall_auc"]),
            "today_overall_auc": float(row["+today_overall_auc"]),
        })
    (args.run_dir / "abs_univariate.json").write_text(
        json.dumps(abs_rows, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    baseline_abs_map = {r["feature"]: r["baseline_abs_lv3_auc"] for r in abs_rows}
    today_abs_map = {r["feature"]: r["today_abs_lv3_auc"] for r in abs_rows}

    baseline_pool_pre = [f for f in FEATURES_320 if baseline_abs_map[f] >= ABS_AUC_CUTOFF]
    today_pool_pre = [f for f in FEATURES_320 if today_abs_map[f] >= ABS_AUC_CUTOFF]

    # -- 2) stop_segment diagnosis
    stop_diag = diagnose_stop_segment(existing, today_macro)
    diag_lines = [
        "# mouse_stop_segment_count 분포 진단",
        "",
        "raw lv3_auc = 0.0 (065547) 의 원인을 NaN imputation 가설로 검토.",
        "ticket 320 univariate 평가는 `SimpleImputer(strategy='median')` 사용 → ",
        "한 그룹이 거의 전체 NaN 이면 median 으로 collapse 되어 LogReg AUC 가 0 또는 1 의 극단에 위치할 수 있음.",
        "",
        "| group | n | non_null | null | non_null_ratio | median | min | max |",
        "| --- | --- | --- | --- | --- | --- | --- | --- |",
    ]
    for name in ["human", "lv2_macro", "lv3_balabit_kde", "today_macro"]:
        s = stop_diag[name]
        diag_lines.append(
            f"| {name} | {s['n']} | {s['non_null_count']} | {s['null_count']} | "
            f"{s['non_null_ratio']:.3f} | {fmt_or_none(s['median'])} | "
            f"{fmt_or_none(s['min'])} | {fmt_or_none(s['max'])} |"
        )
    diag_lines.append("")
    null_ratios = {name: stop_diag[name]["null_count"] / stop_diag[name]["n"]
                   for name in stop_diag if stop_diag[name]["n"] > 0}
    high_null = [g for g, r in null_ratios.items() if r >= 0.5]
    low_null = [g for g, r in null_ratios.items() if r < 0.5]
    is_artifact = bool(high_null) and bool(low_null)
    diag_lines.append("## 판정")
    diag_lines.append("")
    diag_lines.append(f"- null 비율 >= 50% group: {high_null if high_null else '(none)'}")
    diag_lines.append(f"- null 비율 <  50% group: {low_null if low_null else '(none)'}")
    diag_lines.append("")
    if is_artifact:
        diag_lines.append("- 일부 group 은 high-null, 다른 group 은 low-null → **NaN imputation 아티팩트 강한 의심**.")
        diag_lines.append("- median imputation 시 high-null group 은 training median 으로 collapse 되어, low-null group 의 실제 분포와 인공적으로 분리됨.")
        diag_lines.append("- LogReg 가 이 인공 분리를 학습하면 raw AUC 가 0 또는 1 의 극단으로 편향됨 (방향은 high-null group 의 라벨에 따라 결정).")
        diag_lines.append("- XGBoost 는 NaN native 처리이므로 cleaned pool 에 포함시켜도 LogReg AUC 와 의미는 다름. 단, 학습 시 NaN split 패턴이 trial 인구 분포 (e.g., Balabit 가 human 전부) 에 의존하므로 OOD generalization 보장 X.")
    else:
        diag_lines.append("- 그룹별 NaN 분포에서 명확한 imputation 아티팩트 패턴은 보이지 않음.")
        diag_lines.append("- raw lv3_auc 의 극단값 원인은 추가 조사 필요.")
    (args.run_dir / "stop_segment_diagnosis.md").write_text(
        "\n".join(diag_lines) + "\n", encoding="utf-8"
    )

    # -- 3) Correlation pruning per condition
    baseline_pool, baseline_log = prune_by_correlation(
        baseline_pool_pre, baseline_train, baseline_abs_map, CORR_CUTOFF
    )
    today_pool, today_log = prune_by_correlation(
        today_pool_pre, today_train, today_abs_map, CORR_CUTOFF
    )

    union_pool = sorted(set(baseline_pool_pre) | set(today_pool_pre))
    baseline_corr = baseline_train[union_pool].corr() if union_pool else pd.DataFrame()
    today_corr = today_train[union_pool].corr() if union_pool else pd.DataFrame()

    corr_md = ["# Correlation pruning log", "", f"cutoff: |r| >= {CORR_CUTOFF}", ""]
    corr_md.append("## Pre-pruning pool (abs AUC >= 0.65)")
    corr_md.append("")
    corr_md.append(f"- baseline (n={len(baseline_pool_pre)}): {baseline_pool_pre}")
    corr_md.append(f"- +today (n={len(today_pool_pre)}): {today_pool_pre}")
    corr_md.append("")
    corr_md.append("## High-correlation pairs (|r| >= 0.85) on union pool")
    corr_md.append("")
    corr_md.append("| feat1 | feat2 | |r| (baseline_train) | |r| (today_train) |")
    corr_md.append("| --- | --- | --- | --- |")
    found_pair = False
    for i, f1 in enumerate(union_pool):
        for f2 in union_pool[i + 1 :]:
            r_b = baseline_corr.loc[f1, f2] if not baseline_corr.empty else float("nan")
            r_t = today_corr.loc[f1, f2] if not today_corr.empty else float("nan")
            r_b_abs = abs(r_b) if pd.notna(r_b) else float("nan")
            r_t_abs = abs(r_t) if pd.notna(r_t) else float("nan")
            hit_b = pd.notna(r_b_abs) and r_b_abs >= CORR_CUTOFF
            hit_t = pd.notna(r_t_abs) and r_t_abs >= CORR_CUTOFF
            if hit_b or hit_t:
                found_pair = True
                rb = f"{r_b_abs:.4f}" if pd.notna(r_b_abs) else "—"
                rt = f"{r_t_abs:.4f}" if pd.notna(r_t_abs) else "—"
                corr_md.append(f"| {f1} | {f2} | {rb} | {rt} |")
    if not found_pair:
        corr_md.append("| (no pairs above cutoff) | | | |")
    corr_md.append("")
    corr_md.append("## Pruning decisions — baseline")
    corr_md.append("")
    if baseline_log:
        corr_md.append("| dropped | kept | |r| | kept abs AUC | dropped abs AUC |")
        corr_md.append("| --- | --- | --- | --- | --- |")
        for entry in baseline_log:
            corr_md.append(
                f"| {entry['dropped']} | {entry['kept']} | {entry['abs_corr']:.4f} | "
                f"{entry['kept_abs_auc']:.4f} | {entry['dropped_abs_auc']:.4f} |"
            )
    else:
        corr_md.append("(no pruning needed)")
    corr_md.append("")
    corr_md.append("## Pruning decisions — +today")
    corr_md.append("")
    if today_log:
        corr_md.append("| dropped | kept | |r| | kept abs AUC | dropped abs AUC |")
        corr_md.append("| --- | --- | --- | --- | --- |")
        for entry in today_log:
            corr_md.append(
                f"| {entry['dropped']} | {entry['kept']} | {entry['abs_corr']:.4f} | "
                f"{entry['kept_abs_auc']:.4f} | {entry['dropped_abs_auc']:.4f} |"
            )
    else:
        corr_md.append("(no pruning needed)")
    corr_md.append("")
    corr_md.append("## Cleaned pool")
    corr_md.append("")
    corr_md.append(f"- baseline (n={len(baseline_pool)}): {baseline_pool}")
    corr_md.append(f"- +today (n={len(today_pool)}): {today_pool}")
    (args.run_dir / "correlation_pruning_log.md").write_text(
        "\n".join(corr_md) + "\n", encoding="utf-8"
    )

    # -- 4) XGBoost retrain
    baseline_result = train_xgb("baseline", baseline_pool, baseline_train, baseline_test, lv3_eval)
    today_result = train_xgb("+today", today_pool, today_train, today_test, lv3_eval)
    mv_rows = [baseline_result, today_result]
    (args.run_dir / "multivariate_abs.json").write_text(
        json.dumps(mv_rows, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    # -- 5) Report
    delta_overall = baseline_result["overall_auc"] - BASELINE_065547["overall_auc"]
    delta_lv3 = baseline_result["lv3_auc"] - BASELINE_065547["lv3_auc"]
    report = [
        "# Ticket 319 — abs AUC 재산출 + correlation pruning",
        "",
        "## 방식",
        "",
        "- abs_auc = |raw_auc - 0.5| + 0.5",
        f"- abs cutoff: {ABS_AUC_CUTOFF}",
        f"- correlation pruning cutoff: |r| >= {CORR_CUTOFF}",
        f"- XGBoost: n_estimators=400, max_depth=3, learning_rate=0.03, random_state={RANDOM_STATE}",
        "",
        "## abs lv3 AUC (16 feature)",
        "",
        "| feature | baseline_raw | baseline_abs | +today_raw | +today_abs |",
        "| --- | --- | --- | --- | --- |",
    ]
    for row in abs_rows:
        report.append(
            f"| {row['feature']} | {row['baseline_raw_lv3_auc']:.4f} | "
            f"{row['baseline_abs_lv3_auc']:.4f} | {row['today_raw_lv3_auc']:.4f} | "
            f"{row['today_abs_lv3_auc']:.4f} |"
        )
    report += [
        "",
        "## Cutoff 통과 pre-pruning",
        "",
        f"- baseline (n={len(baseline_pool_pre)}): {baseline_pool_pre}",
        f"- +today (n={len(today_pool_pre)}): {today_pool_pre}",
        "",
        "## Cleaned pool (correlation pruning 후)",
        "",
        f"- baseline (n={len(baseline_pool)}): {baseline_pool}",
        f"- +today (n={len(today_pool)}): {today_pool}",
        "",
        "## XGBoost 결과",
        "",
        "| condition | feature_count | overall_auc | lv3_auc |",
        "| --- | --- | --- | --- |",
    ]
    for r in mv_rows:
        report.append(
            f"| {r['condition']} | {r['feature_count']} | "
            f"{r['overall_auc']:.4f} | {r['lv3_auc']:.4f} |"
        )
    report += [
        "",
        "## 065547 baseline (2 feature) 비교",
        "",
        "| metric | 065547 baseline | abs baseline | delta |",
        "| --- | --- | --- | --- |",
        f"| feature_count | {BASELINE_065547['feature_count']} | {baseline_result['feature_count']} | "
        f"{baseline_result['feature_count'] - BASELINE_065547['feature_count']:+d} |",
        f"| overall_auc | {BASELINE_065547['overall_auc']:.4f} | "
        f"{baseline_result['overall_auc']:.4f} | {delta_overall:+.4f} |",
        f"| lv3_auc | {BASELINE_065547['lv3_auc']:.4f} | "
        f"{baseline_result['lv3_auc']:.4f} | {delta_lv3:+.4f} |",
        "",
        f"- 065547 features: {BASELINE_065547['features']}",
        f"- abs cleaned baseline features: {baseline_pool}",
        "",
        "## mouse_stop_segment_count",
        "",
        "- stop_segment_diagnosis.md 참조.",
        "",
        "## 파일",
        "",
        "- abs_univariate.json",
        "- stop_segment_diagnosis.md",
        "- correlation_pruning_log.md",
        "- multivariate_abs.json",
        "- report_abs.md (this)",
    ]
    (args.run_dir / "report_abs.md").write_text(
        "\n".join(report) + "\n", encoding="utf-8"
    )

    # -- console echo
    print("=== run_dir ===")
    print(args.run_dir)
    print()
    print("=== abs cutoff pass (pre-pruning) ===")
    print(f"baseline: {baseline_pool_pre}")
    print(f"+today: {today_pool_pre}")
    print()
    print("=== cleaned pool ===")
    print(f"baseline: {baseline_pool}")
    print(f"+today: {today_pool}")
    print()
    print("=== XGBoost ===")
    for r in mv_rows:
        print(
            f"{r['condition']}: overall_auc={r['overall_auc']:.4f} "
            f"lv3_auc={r['lv3_auc']:.4f} (n={r['feature_count']})"
        )
    print()
    print("=== vs 065547 baseline ===")
    print(
        f"065547: overall={BASELINE_065547['overall_auc']:.4f} "
        f"lv3={BASELINE_065547['lv3_auc']:.4f}"
    )
    print(
        f"  abs baseline: overall={baseline_result['overall_auc']:.4f} "
        f"lv3={baseline_result['lv3_auc']:.4f}"
    )
    print(f"  delta_overall={delta_overall:+.4f} delta_lv3={delta_lv3:+.4f}")
    print()
    print("=== stop_segment_count null ratios ===")
    for name in ["human", "lv2_macro", "lv3_balabit_kde", "today_macro"]:
        s = stop_diag[name]
        nr = s["null_count"] / s["n"] if s["n"] else 0.0
        print(
            f"  {name}: n={s['n']} null={s['null_count']} ({nr:.2%}) "
            f"median={fmt_or_none(s['median'])}"
        )


if __name__ == "__main__":
    main()